## Prerequisites:

Installed following packages:
```shell
uv pip install torch ipykernel tensorflow accelerate datasets evaluate timm
```

I started with Microsoft Phi3, Phi3.5 and Phi-4 models. Unfortunately, these all failed with dependency, strict version requirements of torch transformers packages and compatibility issues, requires deprecated torch_dtype and so on. I decided to try the Qwen and it worked. 

In [6]:
!du -h --max-depth=2 ~/.cache/huggingface

96K	/home/batto/.cache/huggingface/hub/models--unsloth--Phi-4-mini-instruct-bnb-4bit
40K	/home/batto/.cache/huggingface/hub/datasets--ag_news
7.2G	/home/batto/.cache/huggingface/hub/models--microsoft--Phi-3-mini-4k-instruct
7.2G	/home/batto/.cache/huggingface/hub/models--microsoft--Phi-3.5-mini-instruct
96K	/home/batto/.cache/huggingface/hub/models--Unsloth--Phi-4-mini-instruct-bnb-4bit
7.2G	/home/batto/.cache/huggingface/hub/models--microsoft--Phi-4-mini-instruct
7.6G	/home/batto/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507
724K	/home/batto/.cache/huggingface/hub/models--bert-base-uncased
36K	/home/batto/.cache/huggingface/hub/.locks
29G	/home/batto/.cache/huggingface/hub
980K	/home/batto/.cache/huggingface/modules/transformers_modules
984K	/home/batto/.cache/huggingface/modules
4.0K	/home/batto/.cache/huggingface/datasets
1.5M	/home/batto/.cache/huggingface/xet/logs
60K	/home/batto/.cache/huggingface/xet/https___cas_serv-tGqkUaZf_CBPHQ6h
1.6M	/home/batto/.cache/hugging

In [7]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: NVIDIA GeForce RTX 5070 Ti


## HuggingFace model loading

The source code is copied from Chapter 1 "Hands-on Language Models" Jay Alammar, Maarten Grootendorst Oreilly Media Inc.,

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# model_path = "microsoft/Phi-4-mini-instruct"
model_path = "unsloth/Qwen3-4B-Thinking-2507-FP8"
# model_path = "meta-llama/Llama-3.2-3B-Instruct"
# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="cuda",
    dtype="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [2]:
model_inputs = tokenizer(["The secret to be happy is"], return_tensors="pt").to(model.device)

The pipeline encapsulates the model, tokenizer and text completion

In [12]:
from transformers import pipeline

# Create a pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)
generation_args = {
    "max_new_tokens": 500,
    "return_full_text": False,
    "do_sample": False,
}
 

Device set to use cuda


In [13]:
prompt = "Give me a short guidance to being happy."
messages = [
    {"role": "user", "content": prompt}
]
output = pipe(messages, **generation_args)
print(output[0]['generated_text'])

Okay, the user asked for "a short guidance to being happy." Hmm, they probably want something quick and practical, not a long essay. Let me think about what truly matters for happiness without being too fluffy.  

First, I should avoid clichés like "just be positive" because that's not helpful for everyone. Real happiness is deeper—connected to small, actionable steps. The user might be stressed or overwhelmed, so I'll keep it simple and grounded.  

I recall research shows happiness comes from presence, relationships, and meaning. But "short" is key here. Maybe 3-4 bullet points? They said "guidance," so it should feel like a gentle nudge, not a lecture.  

Also, the user didn't specify their situation—could be anyone. Better keep it universal: no assumptions about their life stage or problems. No jargon.  

*Brainstorming points:*  
- **Presence**: People forget to be here right now. Mindfulness is huge but I'll phrase it as "savor small moments" so it's not clinical.  
- **Connectio

In [14]:
prompt = "Explain who was Odysseus."

# Tokenize the input prompt
inputs = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# Generate the text
generation_output = model.generate(
  input_ids=inputs,
  max_new_tokens=400,

)

# Print the output
print(tokenizer.decode(generation_output[0]))

Explain who was Odysseus. What were his major achievements and how did he fit into the story of the Trojan War? Also, how did Odysseus' personality traits influence his actions during the journey home? Can you provide examples of his cleverness and bravery? Additionally, how does Odysseus' journey home reflect the theme of the *Odyssey*? How does the *Odyssey* relate to the *Iliad*? What is the significance of the *Odyssey* in ancient Greek literature? How does the *Odyssey* compare to other ancient literature? What does the *Odyssey* tell us about ancient Greek society? What is the significance of the *Odyssey* in modern times? Can you also explain the role of the *Odyssey* in the *Odyssey* (the epic itself)? Finally, what is the significance of the *Odyssey* in the context of the *Iliad*?

Okay, the user has asked a comprehensive set of questions about Odysseus and the *Odyssey*. Let me break this down step by step. 

First, I need to recall who Odysseus was. He's the king of Ithaca,

In [15]:
print(inputs)

tensor([[  840, 20772,   879,   572, 24560,  1047,   325,   355,    13]],
       device='cuda:0')


In [18]:
for id in inputs[0]:
   print(f"{id}: {tokenizer.decode(id)}")

840: Ex
20772: plain
879:  who
572:  was
24560:  Od
1047: ys
325: se
355: us
13: .


In [16]:
text = """

English and CAPITALIZATION

🎵鸟
show_tokens False None elif == >= else: two tabs:" " Three tabs: "   "

12.0*50=600

"""

In [17]:
colors_list = [
    '102;194;165', '252;141;98', '141;160;203', 
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' + 
            tokenizer.decode(t) + 
            '\x1b[0m', 
            end=' '
        )

**Special tokens** (taken from Hands on Large Language Models book from Oreilly Media Inc.,):

- unk_token [UNK]
  - An unknown token that the tokenizer has no specific encoding for.
- sep_token [SEP]
  - A separator that enables certain tasks that require giving the model two texts (in these cases, the model is called a cross-encoder). 
- pad_token [PAD]
  - A padding token used to pad unused positions in the model’s input (as the model expects a certain length of input, its context-size).
- cls_token [CLS]
  - A special classification token for classification tasks
- mask_token [MASK]
  - A masking token used to hide tokens during the training process.

In [18]:
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content) # no opening <think> tag
print("content:", content)

thinking content: Okay, the user asked for a short introduction to large language models. Hmm, they probably want something concise but informative—maybe they're a beginner in AI or just curious about what LLMs are. I should avoid jargon overload since they specified "short."  

First, I'll define LLMs clearly: they're AI models trained on massive text data to understand and generate human-like language. Key points to hit: scale (huge datasets), training (self-supervised learning), and capabilities (text generation, reasoning, coding).  

Wait—should I mention specific examples? Yeah, naming models like GPT-3 or Gemini makes it concrete. But keep it brief; the user said "short." Also, gotta clarify they're *not* magic—they need good prompts and have limitations (bias, hallucinations).  

User might wonder why LLMs matter. I'll add real-world impact: writing, coding, translation. But no tangents! Stay focused.  

Oh, and emphasize the "why": they process language patterns to mimic human

In [19]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False  # Disables thinking mode
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content) # no opening <think> tag
print("content:", content)

thinking content: Okay, the user asked for a short introduction to large language models. Let me start by recalling what I know. LLMs are a type of AI model that can generate human-like text. They're trained on huge amounts of data, right? 

Hmm, the user might be someone new to AI, maybe a student or a professional looking to get a quick overview. They want something concise but informative. I should avoid jargon as much as possible. 

First, I need to define what an LLM is. Mention that it's a deep learning model with a lot of parameters. But wait, parameters can be confusing. Maybe say "billions or trillions of parameters" to give a sense of scale. 

Key characteristics: trained on massive text datasets, learns patterns and language structures. They can do things like answer questions, write stories, code. Should I mention specific examples? Like ChatGPT, Gemini, Claude? Yeah, that makes it concrete. 

Also, the user might not know the difference between LLMs and other models. Maybe